In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Locate project root

project_root = Path.cwd()

if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent

raw_path = project_root / "data" / "raw"
interim_path = project_root / "data" / "interim"

interim_path.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data path:", raw_path)
print("Interim data path:", interim_path)

Project root: c:\Users\ANKIT TRIPATHI\Desktop\SaaS-Revenue-Subscription-Analytics
Raw data path: c:\Users\ANKIT TRIPATHI\Desktop\SaaS-Revenue-Subscription-Analytics\data\raw
Interim data path: c:\Users\ANKIT TRIPATHI\Desktop\SaaS-Revenue-Subscription-Analytics\data\interim


In [3]:
datasets = {
    "plans": pd.read_csv(raw_path / "plans.csv"),
    "customers": pd.read_csv(raw_path / "customers.csv"),
    "subscriptions": pd.read_csv(raw_path / "subscriptions.csv"),
    "subscription_months": pd.read_csv(
        raw_path / "subscription_months.csv"
    ),
    "invoices": pd.read_csv(raw_path / "invoices.csv")
}

print("DATASETS LOADED SUCCESSFULLY")
print("=" * 50)

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

DATASETS LOADED SUCCESSFULLY
plans: 10 rows, 11 columns
customers: 1200 rows, 12 columns
subscriptions: 1381 rows, 12 columns
subscription_months: 19015 rows, 15 columns
invoices: 11257 rows, 17 columns


In [4]:
total_rows = sum(len(df) for df in datasets.values())

total_columns = sum(
    len(df.columns) for df in datasets.values()
)

print("Total records:", total_rows)
print("Total columns:", total_columns)

assert total_rows == 32863, "Unexpected total row count!"
assert total_columns == 67, "Unexpected total column count!"

print("RESULT: PASS - All datasets loaded correctly.")

Total records: 32863
Total columns: 67
RESULT: PASS - All datasets loaded correctly.


In [5]:
# Step 3.7 Part B.1 — Create cleaning copies

cleaned_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

print("Working copies created successfully!")

for name, df in cleaned_datasets.items():
    print(f"{name}: {df.shape}")

Working copies created successfully!
plans: (10, 11)
customers: (1200, 12)
subscriptions: (1381, 12)
subscription_months: (19015, 15)
invoices: (11257, 17)


In [8]:
# Step 3.7 Part B.2 — Correct Date Standardization

date_formats = {
    "plans": {
        "launched_date": "%d-%m-%Y"
    },
    "customers": {
        "signup_date": "%d-%m-%Y",
        "churn_date": "%d-%m-%Y",
        "cohort_month": "%Y-%m"
    },
    "subscriptions": {
        "start_date": "%d-%m-%Y",
        "end_date": "%d-%m-%Y"
    },
    "subscription_months": {
        "period_month": "%Y-%m",
        "month_start_date": "%d-%m-%Y",
        "cohort_month": "%Y-%m"
    },
    "invoices": {
        "invoice_date": "%d-%m-%Y",
        "invoice_month": "%Y-%m",
        "billing_period_start": "%d-%m-%Y",
        "billing_period_end": "%d-%m-%Y"
    }
}

date_conversion_results = []

for table_name, columns in date_formats.items():

    df = cleaned_datasets[table_name]

    for column, date_format in columns.items():

        # Always parse from the original raw-loaded values
        original = datasets[table_name][column]

        parsed = pd.to_datetime(
            original,
            format=date_format,
            errors="coerce"
        )

        new_invalid = int(
            (original.notna() & parsed.isna()).sum()
        )

        df[column] = parsed

        date_conversion_results.append({
            "Dataset": table_name,
            "Column": column,
            "Format": date_format,
            "Data_Type": str(df[column].dtype),
            "New_Invalid_Dates": new_invalid
        })

date_conversion_summary = pd.DataFrame(
    date_conversion_results
)

print("CORRECTED DATE STANDARDIZATION RESULTS")
print("=" * 85)

print(date_conversion_summary.to_string(index=False))

CORRECTED DATE STANDARDIZATION RESULTS
            Dataset               Column   Format      Data_Type  New_Invalid_Dates
              plans        launched_date %d-%m-%Y datetime64[us]                  0
          customers          signup_date %d-%m-%Y datetime64[us]                  0
          customers           churn_date %d-%m-%Y datetime64[us]                  0
          customers         cohort_month    %Y-%m datetime64[us]                  0
      subscriptions           start_date %d-%m-%Y datetime64[us]                  0
      subscriptions             end_date %d-%m-%Y datetime64[us]                  0
subscription_months         period_month    %Y-%m datetime64[us]                  0
subscription_months     month_start_date %d-%m-%Y datetime64[us]                  0
subscription_months         cohort_month    %Y-%m datetime64[us]                  0
           invoices         invoice_date %d-%m-%Y datetime64[us]                  0
           invoices        invoice_mo

In [9]:
print("DATE CONVERSION VERIFICATION")
print("=" * 55)

print(
    "Plan launched date:",
    cleaned_datasets["plans"]["launched_date"].iloc[0]
)

print(
    "Monthly start date (row 2):",
    cleaned_datasets["subscription_months"]["month_start_date"].iloc[1]
)

print(
    "Invoice date (row 1):",
    cleaned_datasets["invoices"]["invoice_date"].iloc[0]
)

print(
    "Customer cohort:",
    cleaned_datasets["customers"]["cohort_month"].iloc[0]
)

print(
    "\nTotal invalid dates:",
    int(date_conversion_summary["New_Invalid_Dates"].sum())
)

DATE CONVERSION VERIFICATION
Plan launched date: 2022-01-15 00:00:00
Monthly start date (row 2): 2024-02-01 00:00:00
Invoice date (row 1): 2024-01-02 00:00:00
Customer cohort: 2024-01-01 00:00:00

Total invalid dates: 0


In [10]:
# Step 3.7 Part B.3.1 — Numeric Data Type Inspection

numeric_type_results = []

for table_name, df in cleaned_datasets.items():

    for column in df.columns:

        numeric_type_results.append({
            "Dataset": table_name,
            "Column": column,
            "Data_Type": str(df[column].dtype),
            "Is_Numeric": pd.api.types.is_numeric_dtype(
                df[column]
            )
        })

column_type_summary = pd.DataFrame(numeric_type_results)

print("DATA TYPE INSPECTION")
print("=" * 70)

print(column_type_summary.to_string(index=False))

DATA TYPE INSPECTION
            Dataset               Column      Data_Type  Is_Numeric
              plans              plan_id            str       False
              plans            plan_name            str       False
              plans                 tier            str       False
              plans     billing_interval            str       False
              plans       list_price_usd          int64        True
              plans    monthly_price_usd        float64        True
              plans       billing_months          int64        True
              plans       seats_included          int64        True
              plans extra_seat_price_usd          int64        True
              plans            is_active            str       False
              plans        launched_date datetime64[us]       False
          customers          customer_id            str       False
          customers         company_name            str       False
          customers        

In [11]:
# Step 3.7 Part B.3.2 — Financial Numeric Type Validation

expected_numeric_columns = {
    "plans": [
        "monthly_price_usd"
    ],
    "subscription_months": [
        "mrr_usd",
        "previous_mrr_usd",
        "mrr_delta_usd"
    ],
    "invoices": [
        "amount_usd",
        "discount_usd",
        "subtotal_usd",
        "tax_usd",
        "total_usd",
        "days_to_payment"
    ]
}

numeric_validation_results = []

for table_name, columns in expected_numeric_columns.items():

    df = cleaned_datasets[table_name]

    for column in columns:

        if column not in df.columns:
            numeric_validation_results.append({
                "Dataset": table_name,
                "Column": column,
                "Data_Type": "MISSING COLUMN",
                "Is_Numeric": False,
                "Missing_Values": None
            })
            continue

        numeric_validation_results.append({
            "Dataset": table_name,
            "Column": column,
            "Data_Type": str(df[column].dtype),
            "Is_Numeric": bool(
                pd.api.types.is_numeric_dtype(df[column])
            ),
            "Missing_Values": int(df[column].isna().sum())
        })

numeric_validation_summary = pd.DataFrame(
    numeric_validation_results
)

print("FINANCIAL NUMERIC TYPE VALIDATION")
print("=" * 80)

print(numeric_validation_summary.to_string(index=False))

FINANCIAL NUMERIC TYPE VALIDATION
            Dataset            Column Data_Type  Is_Numeric  Missing_Values
              plans monthly_price_usd   float64        True               0
subscription_months           mrr_usd   float64        True               0
subscription_months  previous_mrr_usd   float64        True               0
subscription_months     mrr_delta_usd   float64        True               0
           invoices        amount_usd   float64        True               0
           invoices      discount_usd   float64        True               0
           invoices      subtotal_usd   float64        True               0
           invoices           tax_usd   float64        True               0
           invoices         total_usd   float64        True               0
           invoices   days_to_payment   float64        True             309


In [12]:
# Step 3.7 Part B.3.3 — Final Numeric Type Check

failed_columns = numeric_validation_summary[
    ~numeric_validation_summary["Is_Numeric"]
]

print("NUMERIC TYPE VERIFICATION SUMMARY")
print("=" * 60)

print(
    "Financial columns checked:",
    len(numeric_validation_summary)
)

print(
    "Columns with incorrect numeric types:",
    len(failed_columns)
)

if failed_columns.empty:
    print("RESULT: PASS - All checked financial columns are numeric.")
else:
    print("RESULT: REVIEW REQUIRED")
    print(failed_columns.to_string(index=False))

NUMERIC TYPE VERIFICATION SUMMARY
Financial columns checked: 10
Columns with incorrect numeric types: 0
RESULT: PASS - All checked financial columns are numeric.


In [13]:
# Step 3.7 Part C.1 — Categorical Column Inspection

categorical_results = []

for table_name, df in cleaned_datasets.items():

    categorical_columns = df.select_dtypes(
        include=["object", "string", "category"]
    ).columns

    for column in categorical_columns:

        categorical_results.append({
            "Dataset": table_name,
            "Column": column,
            "Unique_Values": df[column].nunique(dropna=True),
            "Missing_Values": int(df[column].isna().sum())
        })

categorical_summary = pd.DataFrame(categorical_results)

print("CATEGORICAL COLUMN INSPECTION")
print("=" * 75)

print(categorical_summary.to_string(index=False))

CATEGORICAL COLUMN INSPECTION
            Dataset              Column  Unique_Values  Missing_Values
              plans             plan_id             10               0
              plans           plan_name             10               0
              plans                tier              4               0
              plans    billing_interval              2               0
              plans           is_active              2               0
          customers         customer_id           1200               0
          customers        company_name           1200               0
          customers             country             20               0
          customers              region              6               0
          customers            industry             12               0
          customers             segment              3               0
          customers      employees_band              5               0
          customers acquisition_channel        

In [14]:
# Step 3.7 Part C.1.2 — Business Category Inspection

business_categories = {
    "customers": ["customer_status"],
    "subscriptions": ["subscription_status"],
    "subscription_months": ["movement_type"],
    "invoices": ["invoice_status"]
}

for table_name, columns in business_categories.items():

    df = cleaned_datasets[table_name]

    for column in columns:

        print(f"\n{table_name}.{column}")
        print("-" * 55)

        print(
            df[column]
            .value_counts(dropna=False)
            .to_string()
        )


customers.customer_status
-------------------------------------------------------
customer_status
Active     855
Churned    345

subscriptions.subscription_status
-------------------------------------------------------
subscription_status
Active        855
Churned       368
Upgraded      115
Downgraded     43

subscription_months.movement_type
-------------------------------------------------------
movement_type
Retained        16143
New              1200
Expansion        1023
Churn             368
Contraction       258
Reactivation       23

invoices.invoice_status
-------------------------------------------------------
invoice_status
Paid        10316
Overdue       525
Failed        309
Refunded      107


In [15]:
# Step 3.7 Part C.1.3 — Detect Whitespace Issues

whitespace_results = []

for table_name, columns in business_categories.items():

    df = cleaned_datasets[table_name]

    for column in columns:

        values = df[column].dropna().astype(str)

        extra_spaces = int(
            (values != values.str.strip()).sum()
        )

        whitespace_results.append({
            "Dataset": table_name,
            "Column": column,
            "Extra_Space_Records": extra_spaces
        })

whitespace_summary = pd.DataFrame(whitespace_results)

print("CATEGORICAL WHITESPACE VALIDATION")
print("=" * 65)

print(whitespace_summary.to_string(index=False))

print(
    "\nTotal whitespace issues:",
    int(whitespace_summary["Extra_Space_Records"].sum())
)

CATEGORICAL WHITESPACE VALIDATION
            Dataset              Column  Extra_Space_Records
          customers     customer_status                    0
      subscriptions subscription_status                    0
subscription_months       movement_type                    0
           invoices      invoice_status                    0

Total whitespace issues: 0


In [16]:
# Step 3.7 Part C.1.4 — Standardize Business Categories

for table_name, columns in business_categories.items():

    df = cleaned_datasets[table_name]

    for column in columns:

        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.title()
        )

print("Business category standardization completed.")

for table_name, columns in business_categories.items():
    for column in columns:

        print(f"\n{table_name}.{column}")

        print(
            cleaned_datasets[table_name][column]
            .value_counts(dropna=False)
            .to_string()
        )

Business category standardization completed.

customers.customer_status
customer_status
Active     855
Churned    345

subscriptions.subscription_status
subscription_status
Active        855
Churned       368
Upgraded      115
Downgraded     43

subscription_months.movement_type
movement_type
Retained        16143
New              1200
Expansion        1023
Churn             368
Contraction       258
Reactivation       23

invoices.invoice_status
invoice_status
Paid        10316
Overdue       525
Failed        309
Refunded      107


In [17]:
# Step 3.7 Part C.1.5 — Record Count Verification

for table_name in datasets:

    original_count = len(datasets[table_name])
    cleaned_count = len(cleaned_datasets[table_name])

    print(
        f"{table_name}: "
        f"Original={original_count}, "
        f"Cleaned={cleaned_count}"
    )

    assert original_count == cleaned_count, (
        f"Row count mismatch in {table_name}"
    )

print("\nRESULT: PASS - No records were removed.")

plans: Original=10, Cleaned=10
customers: Original=1200, Cleaned=1200
subscriptions: Original=1381, Cleaned=1381
subscription_months: Original=19015, Cleaned=19015
invoices: Original=11257, Cleaned=11257

RESULT: PASS - No records were removed.


In [18]:
# Step 3.7 Part C.2.1 — Dataset Structure Validation

structure_results = []

for table_name in datasets:

    original = datasets[table_name]
    cleaned = cleaned_datasets[table_name]

    structure_results.append({
        "Dataset": table_name,
        "Original_Rows": len(original),
        "Cleaned_Rows": len(cleaned),
        "Original_Columns": len(original.columns),
        "Cleaned_Columns": len(cleaned.columns),
        "Same_Column_Names": (
            original.columns.tolist()
            == cleaned.columns.tolist()
        )
    })

structure_summary = pd.DataFrame(structure_results)

print("CLEANED DATASET STRUCTURE VALIDATION")
print("=" * 85)

print(structure_summary.to_string(index=False))

CLEANED DATASET STRUCTURE VALIDATION
            Dataset  Original_Rows  Cleaned_Rows  Original_Columns  Cleaned_Columns  Same_Column_Names
              plans             10            10                11               11               True
          customers           1200          1200                12               12               True
      subscriptions           1381          1381                12               12               True
subscription_months          19015         19015                15               15               True
           invoices          11257         11257                17               17               True


In [19]:
# Step 3.7 Part C.2.2 — Primary Key Validation

primary_keys = {
    "plans": "plan_id",
    "customers": "customer_id",
    "subscriptions": "subscription_id",
    "subscription_months": "subscription_month_id",
    "invoices": "invoice_id"
}

key_results = []

for table_name, key_column in primary_keys.items():

    original = datasets[table_name]
    cleaned = cleaned_datasets[table_name]

    if key_column not in original.columns:
        key_results.append({
            "Dataset": table_name,
            "Primary_Key": key_column,
            "Status": "COLUMN NOT FOUND"
        })
        continue

    same_values = original[key_column].equals(
        cleaned[key_column]
    )

    duplicate_count = int(
        cleaned[key_column].duplicated().sum()
    )

    missing_count = int(
        cleaned[key_column].isna().sum()
    )

    status = (
        "PASS"
        if same_values
        and duplicate_count == 0
        and missing_count == 0
        else "REVIEW REQUIRED"
    )

    key_results.append({
        "Dataset": table_name,
        "Primary_Key": key_column,
        "Status": status
    })

key_summary = pd.DataFrame(key_results)

print("PRIMARY KEY VALIDATION")
print("=" * 65)

print(key_summary.to_string(index=False))

PRIMARY KEY VALIDATION
            Dataset           Primary_Key           Status
              plans               plan_id             PASS
          customers           customer_id             PASS
      subscriptions       subscription_id             PASS
subscription_months subscription_month_id COLUMN NOT FOUND
           invoices            invoice_id             PASS


In [20]:
# Step 3.7 Part C.2.3 — Missing Value Preservation

missing_results = []

for table_name in datasets:

    original_missing = int(
        datasets[table_name].isna().sum().sum()
    )

    cleaned_missing = int(
        cleaned_datasets[table_name].isna().sum().sum()
    )

    missing_results.append({
        "Dataset": table_name,
        "Original_Missing": original_missing,
        "Cleaned_Missing": cleaned_missing,
        "Match": original_missing == cleaned_missing
    })

missing_summary = pd.DataFrame(missing_results)

print("MISSING VALUE PRESERVATION")
print("=" * 70)

print(missing_summary.to_string(index=False))

print(
    "\nTotal original missing:",
    missing_summary["Original_Missing"].sum()
)

print(
    "Total cleaned missing:",
    missing_summary["Cleaned_Missing"].sum()
)

MISSING VALUE PRESERVATION
            Dataset  Original_Missing  Cleaned_Missing  Match
              plans                 0                0   True
          customers               855              855   True
      subscriptions               855              855   True
subscription_months                 0                0   True
           invoices               309              309   True

Total original missing: 2019
Total cleaned missing: 2019


In [21]:
# Step 3.7 Part C.2.4 — Final Validation

structure_pass = (
    (structure_summary["Original_Rows"]
     == structure_summary["Cleaned_Rows"]).all()
    and
    (structure_summary["Original_Columns"]
     == structure_summary["Cleaned_Columns"]).all()
    and
    structure_summary["Same_Column_Names"].all()
)

keys_pass = (
    key_summary["Status"] == "PASS"
).all()

missing_pass = missing_summary["Match"].all()

print("FINAL CLEANED DATA VALIDATION")
print("=" * 60)

print("Structure validation:", structure_pass)
print("Primary key validation:", keys_pass)
print("Missing value preservation:", missing_pass)

if structure_pass and keys_pass and missing_pass:
    print("\nRESULT: PASS")
else:
    print("\nRESULT: REVIEW REQUIRED")

FINAL CLEANED DATA VALIDATION
Structure validation: True
Primary key validation: False
Missing value preservation: True

RESULT: REVIEW REQUIRED


In [22]:
# Step 3.7 Part D.1 — Prepare interim directory

interim_path = project_root / "data" / "interim"

interim_path.mkdir(parents=True, exist_ok=True)

print("Interim folder:", interim_path)
print("Folder exists:", interim_path.is_dir())

Interim folder: c:\Users\ANKIT TRIPATHI\Desktop\SaaS-Revenue-Subscription-Analytics\data\interim
Folder exists: True


In [23]:
# Step 3.7 Part D.2 — Export cleaned datasets

export_results = []

for table_name, df in cleaned_datasets.items():

    output_file = interim_path / f"{table_name}_cleaned.csv"

    df.to_csv(
        output_file,
        index=False,
        date_format="%Y-%m-%d"
    )

    export_results.append({
        "Dataset": table_name,
        "File": output_file.name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "File_Exists": output_file.is_file()
    })

export_summary = pd.DataFrame(export_results)

print("CLEANED DATASET EXPORT SUMMARY")
print("=" * 85)

print(export_summary.to_string(index=False))

CLEANED DATASET EXPORT SUMMARY
            Dataset                            File  Rows  Columns  File_Exists
              plans               plans_cleaned.csv    10       11         True
          customers           customers_cleaned.csv  1200       12         True
      subscriptions       subscriptions_cleaned.csv  1381       12         True
subscription_months subscription_months_cleaned.csv 19015       15         True
           invoices            invoices_cleaned.csv 11257       17         True


In [24]:
# Step 3.7 Part D.3 — Verify exported CSV files

verification_results = []

for table_name, original_df in cleaned_datasets.items():

    file_path = interim_path / f"{table_name}_cleaned.csv"

    exported_df = pd.read_csv(file_path)

    verification_results.append({
        "Dataset": table_name,
        "Expected_Rows": len(original_df),
        "Exported_Rows": len(exported_df),
        "Expected_Columns": len(original_df.columns),
        "Exported_Columns": len(exported_df.columns),
        "Column_Names_Match": (
            original_df.columns.tolist()
            == exported_df.columns.tolist()
        )
    })

verification_summary = pd.DataFrame(verification_results)

print("CSV EXPORT VERIFICATION")
print("=" * 95)

print(verification_summary.to_string(index=False))

CSV EXPORT VERIFICATION
            Dataset  Expected_Rows  Exported_Rows  Expected_Columns  Exported_Columns  Column_Names_Match
              plans             10             10                11                11                True
          customers           1200           1200                12                12                True
      subscriptions           1381           1381                12                12                True
subscription_months          19015          19015                15                15                True
           invoices          11257          11257                17                17                True


In [25]:
# Step 3.7 Part D.4 — Final export checks

rows_match = (
    verification_summary["Expected_Rows"]
    == verification_summary["Exported_Rows"]
).all()

columns_match = (
    verification_summary["Expected_Columns"]
    == verification_summary["Exported_Columns"]
).all()

names_match = (
    verification_summary["Column_Names_Match"]
).all()

files_exist = export_summary["File_Exists"].all()

print("FINAL CSV EXPORT VALIDATION")
print("=" * 60)

print("All files exist:", files_exist)
print("Row counts match:", rows_match)
print("Column counts match:", columns_match)
print("Column names match:", names_match)

if files_exist and rows_match and columns_match and names_match:
    print("\nRESULT: PASS - All cleaned CSV files exported successfully.")
else:
    print("\nRESULT: REVIEW REQUIRED")

FINAL CSV EXPORT VALIDATION
All files exist: True
Row counts match: True
Column counts match: True
Column names match: True

RESULT: PASS - All cleaned CSV files exported successfully.
